# Resumen — Dispersión, Outliers y Boxplots

> **Para el TP2 de Estadística Descriptiva II**  
> Varianza, desvío estándar, IQR, regla 68-95-99.7, z-score y detección de outliers.


---
## 1. Medidas de Dispersión

Responden a: **¿cuánto se alejan los datos de su centro?**

| Medida | Fórmula (resumen) | Unidades | Robustez ante outliers |
|---|---|---|---|
| **Varianza** | promedio de (x - media)² | unidades² | Baja |
| **Desvío estándar** | √varianza | mismas que los datos | Baja |
| **IQR** | Q3 − Q1 | mismas que los datos | **Alta** |

**Cuándo usar cada una:**
- **Desvío estándar** cuando los datos son aproximadamente simétricos (sin outliers graves).
- **IQR** cuando los datos son asimétricos o tienen outliers; no se deja distorsionar.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

age  = df['Age'].dropna()
fare = df['Fare'].dropna()

print('=== AGE ===')
print(f'  Media  : {age.mean():.2f}')
print(f'  Desvío : {age.std():.2f}')
print(f'  IQR    : {age.quantile(0.75) - age.quantile(0.25):.2f}')

print('\n=== FARE ===')
print(f'  Media  : {fare.mean():.2f}')
print(f'  Desvío : {fare.std():.2f}')
print(f'  IQR    : {fare.quantile(0.75) - fare.quantile(0.25):.2f}')

print('\n=> Fare tiene un desvío enorme (outliers de primera clase).')
print('   Su IQR es mucho más pequeño — el 50% central de las tarifas es acotado.')


---
## 2. Regla 68-95-99.7 (solo para distribuciones normales)

Si los datos siguen una **distribución normal**, se cumple:

| Rango | % de datos esperado |
|---|---|
| media ± 1 desvío | ≈ 68% |
| media ± 2 desvíos | ≈ 95% |
| media ± 3 desvíos | ≈ 99.7% |

Usamos esto para **verificar normalidad**: si los porcentajes reales se alejan mucho de estos valores, la distribución no es normal.


In [ ]:
def regla_68_95(serie, nombre):
    mu    = serie.mean()
    sigma = serie.std()
    print(f'{nombre}:')
    for n, esperado in [(1, 68), (2, 95), (3, 99.7)]:
        real = ((serie >= mu - n*sigma) & (serie <= mu + n*sigma)).mean() * 100
        dif  = abs(real - esperado)
        flag = '  ← se aleja' if dif > 5 else ''
        print(f'  {n} desv.: {real:5.1f}%  (esperado {esperado}%){flag}')
    print()

regla_68_95(age,  'AGE  → aproximadamente normal')
regla_68_95(fare, 'FARE → muy asimétrica, NO normal')


---
## 3. Detección de Outliers

### Método Z-score
Transforma cada valor en 'cuántos desvíos está de la media'.  
Si **|z| > 3** → outlier.

$$z_i = \frac{x_i - \bar{x}}{s}$$

**Limitación:** asume distribución normal. Si los datos son asimétricos, detecta pocos outliers.

### Método IQR (Regla de Tukey)
Calcula límites fuera del 50% central.  
Si **x < Q1 - 1.5·IQR** o **x > Q3 + 1.5·IQR** → outlier.

**Ventaja:** no asume normalidad. Más robusto para datos asimétricos.


In [ ]:
# Z-score
df['z_age']  = (df['Age']  - df['Age'].mean())  / df['Age'].std()
df['z_fare'] = (df['Fare'] - df['Fare'].mean()) / df['Fare'].std()

out_z_age  = df[df['z_age'].abs()  > 3]
out_z_fare = df[df['z_fare'].abs() > 3]

# IQR
def outliers_iqr(serie):
    q1, q3 = serie.quantile(0.25), serie.quantile(0.75)
    iqr = q3 - q1
    return serie[(serie < q1 - 1.5*iqr) | (serie > q3 + 1.5*iqr)]

out_iqr_age  = outliers_iqr(age)
out_iqr_fare = outliers_iqr(fare)

print(f'Método      | Age            | Fare')
print(f'Z-score     | {len(out_z_age):2d} outliers     | {len(out_z_fare):2d} outliers')
print(f'IQR (Tukey) | {len(out_iqr_age):2d} outliers     | {len(out_iqr_fare):2d} outliers')
print()
print('=> Para Fare (asimétrica): IQR detecta muchos más outliers que z-score.')
print('   Para Age (más normal): ambos métodos coinciden más.')


---
## 4. Boxplot — cómo leerlo

Un boxplot resume la distribución con 5 números:

```
         ─────────────
         |           |
 ────────|     50%   |────────   ← la caja es el 50% central
  min*   Q1    Q2   Q3   max*     * límites ajustados por IQR
    ○○○  ← outliers (puntos fuera de los bigotes)
```

| Elemento | Qué representa |
|---|---|
| Borde inferior de la caja | Q1 (P25) |
| Línea dentro de la caja | Mediana (Q2/P50) |
| Borde superior de la caja | Q3 (P75) |
| Bigote inferior | Q1 − 1.5·IQR |
| Bigote superior | Q3 + 1.5·IQR |
| Puntos sueltos | Outliers |

**Caja angosta** → datos concentrados. **Caja ancha** → alta dispersión.  
**Caja descentrada** (la mediana no está al medio de la caja) → distribución asimétrica.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

sns.boxplot(x=age,  ax=axes[0], color='#1C7293')
axes[0].set_title('Edad — distribución casi simétrica\ncaja centrada, pocos outliers')
axes[0].set_xlabel('Edad (años)')

sns.boxplot(x=fare, ax=axes[1], color='#F4A261')
axes[1].set_title('Tarifa — distribución MUY asimétrica\ncaja pequeña, cola larga de outliers')
axes[1].set_xlabel('Tarifa (libras)')

plt.tight_layout()
plt.show()


In [ ]:
# Boxplot por grupo — muy útil para comparar
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(x='Pclass', y='Age',  data=df, ax=axes[0], palette='Blues')
axes[0].set_title('Edad por clase')

sns.boxplot(x='Pclass', y='Fare', data=df, ax=axes[1], palette='Oranges')
axes[1].set_title('Tarifa por clase')
axes[1].set_ylim(0, 300)

plt.tight_layout()
plt.show()

print('Observación: en 1a clase las tarifas tienen MÁS dispersión (caja más alta)')
print('y muchos outliers hacia arriba (pasajeros que pagaron suites de lujo).')


---
## 5. Tabla de resumen rápido

| Pregunta | Herramienta |
|---|---|
| ¿Qué tan dispersos están? | Desvío estándar o IQR |
| ¿Los datos son normales? | Regla 68-95-99.7 o Shapiro-Wilk |
| ¿Hay outliers? (normal) | Z-score (\|z\| > 3) |
| ¿Hay outliers? (asimétrica) | IQR (Tukey) |
| ¿Cómo se ve la distribución? | Boxplot |
| ¿Diferencias entre grupos? | Boxplot comparativo |

> **Tip de examen:** cuando la distribución es asimétrica (como precios, salarios, tarifas), usá siempre **IQR** para detectar outliers y **mediana** como medida central. Nunca uses solo la media + z-score en datos con sesgo fuerte.
